# 4 — prepare_training_data

This is the **first training-stage notebook**.

### Input

```text
app/dataset/output/preprocessing/conditions_model_ready.json
```

### Outputs

```text
app/dataset/output/training/
├── training_examples.json
├── train.json
├── validation.json
└── test.json
```

A generation/split report is also saved under:

```text
app/dataset/review/training_generation_report.json
```

## Important design rules

- Training examples use **only reviewed NHS-derived `symptom_phrases`**.
- No new symptoms or synonyms are invented.
- Different examples are created by selecting different subsets of a condition's existing symptom phrases.
- Phrase-order permutations are **not** treated as new examples, reducing leakage.
- Exact duplicate inputs are merged.
- If the exact same generated symptom input maps to multiple conditions, its labels are merged into a **multi-label example** instead of creating contradictory duplicate rows.
- Every condition label must appear in the training split.
- Validation/test are source-derived synthetic evaluations, not independent clinical datasets.


## 1. Imports, configuration, and paths

In [38]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import itertools
import json
import math
import random
import re
import unicodedata


RANDOM_SEED = 42

# Maximum unique source-grounded examples generated per condition.
MAX_EXAMPLES_PER_CONDITION = 30

# Examples are built from 1 to 5 source symptom phrases.
# The complete symptom set is also included once per condition.
MIN_PHRASES_PER_EXAMPLE = 1
MAX_PHRASES_PER_EXAMPLE = 5

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(
    TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0
) < 1e-9


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "app").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. Expected an app/ folder."
    )


PROJECT_ROOT = find_project_root()

INPUT_PATH = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "output"
    / "preprocessing"
    / "conditions_model_ready.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "output"
    / "training"
)

REVIEW_DIR = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "review"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

ALL_EXAMPLES_PATH = OUTPUT_DIR / "training_examples.json"
TRAIN_PATH = OUTPUT_DIR / "train.json"
VALIDATION_PATH = OUTPUT_DIR / "validation.json"
TEST_PATH = OUTPUT_DIR / "test.json"

REPORT_PATH = REVIEW_DIR / "training_generation_report.json"

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Training output:", OUTPUT_DIR)


Project root: C:\Users\po\Downloads\medical-diagnosis
Input: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_model_ready.json
Training output: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\training


## 2. Load and validate the model-ready conditions

In [39]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_PATH}. "
        "Run 3_prepare_model_data.ipynb first."
    )

with INPUT_PATH.open("r", encoding="utf-8") as f:
    conditions = json.load(f)

if not isinstance(conditions, list):
    raise TypeError(
        "conditions_model_ready.json must contain a list."
    )

required_fields = {
    "condition",
    "symptom_phrases",
    "symptoms_canonical",
    "symptoms_classical",
    "symptoms_bert",
}

missing = []

for index, record in enumerate(conditions):
    absent = required_fields - set(record)

    if absent:
        missing.append(
            {
                "index": index,
                "condition": record.get("condition"),
                "missing": sorted(absent),
            }
        )

if missing:
    raise ValueError(
        "Some records are missing required fields:\n"
        + json.dumps(
            missing[:10],
            ensure_ascii=False,
            indent=2,
        )
    )

condition_names = [
    str(record["condition"]).strip()
    for record in conditions
]

duplicates = [
    name
    for name, count in Counter(condition_names).items()
    if count > 1
]

assert all(condition_names), "Found an empty condition label."
assert not duplicates, f"Duplicate condition labels: {duplicates[:10]}"

print("Model-ready conditions:", len(conditions))
print(
    "Total source symptom phrases:",
    sum(len(r["symptom_phrases"]) for r in conditions),
)
print(
    "Minimum phrases for one condition:",
    min(len(r["symptom_phrases"]) for r in conditions),
)
print(
    "Maximum phrases for one condition:",
    max(len(r["symptom_phrases"]) for r in conditions),
)


Model-ready conditions: 376
Total source symptom phrases: 2799
Minimum phrases for one condition: 1
Maximum phrases for one condition: 28


## 3. Text helpers

The training examples are derived from `symptom_phrases`.

We rebuild the model-specific text from the selected phrases so each generated example can be used by:

- TF-IDF / Logistic Regression
- RNN / LSTM
- BERT / BioBERT

No lemmatization is applied here because the final preprocessing decision was to keep the main dataset non-lemmatized.


In [40]:
TOKEN_RE = re.compile(
    r"[^\W_]+(?:['’-][^\W_]+)*|\d+(?:\.\d+)?",
    flags=re.UNICODE,
)


def normalize_space(text: str) -> str:
    text = unicodedata.normalize(
        "NFC",
        str(text or ""),
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def normalize_classical(text: str) -> str:
    text = normalize_space(text).lower()

    text = re.sub(
        r"[^\w\s'’-]",
        " ",
        text,
        flags=re.UNICODE,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def build_example_text(symptom_phrases):
    phrases = [
        normalize_space(phrase)
        for phrase in symptom_phrases
        if normalize_space(phrase)
    ]

    canonical = " ; ".join(phrases)
    classical = normalize_classical(canonical)
    tokens = TOKEN_RE.findall(classical)

    return {
        "symptoms_canonical": canonical,
        "symptoms_classical": " ".join(tokens),
        "symptoms_tokens": tokens,
        "symptoms_bert": canonical,
    }


def stable_seed(text: str) -> int:
    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return int(
        digest[:16],
        16,
    )


## 4. Generate unique symptom combinations for one condition

For each condition:

- all source symptom phrases remain unchanged
- we generate unique subsets of those phrases
- the complete source symptom set is always included
- we cap the number of examples per condition to avoid combinatorial explosion
- the same combination is never added twice


In [41]:
def generate_phrase_subsets(
    condition,
    symptom_phrases,
    max_examples=MAX_EXAMPLES_PER_CONDITION,
):
    phrases = [
        normalize_space(phrase)
        for phrase in symptom_phrases
        if normalize_space(phrase)
    ]

    # Remove exact duplicate phrases while preserving source order.
    phrases = list(dict.fromkeys(phrases))

    if not phrases:
        return []

    n = len(phrases)
    subsets = []
    seen = set()

    def add_indices(indices):
        signature = tuple(sorted(indices))

        if signature in seen:
            return

        seen.add(signature)

        subsets.append(
            [phrases[i] for i in indices]
        )

    # Always include the complete reviewed symptom set.
    add_indices(tuple(range(n)))

    # Include single symptoms. These are useful for short user inputs.
    # Ambiguous identical inputs will be merged across conditions later.
    for i in range(n):
        if len(subsets) >= max_examples:
            break

        add_indices((i,))

    # Add all small combinations when the search space is small.
    max_k = min(
        MAX_PHRASES_PER_EXAMPLE,
        n,
    )

    possible_count = sum(
        math.comb(n, k)
        for k in range(
            max(MIN_PHRASES_PER_EXAMPLE, 2),
            max_k + 1,
        )
    )

    if possible_count <= 5000:
        all_combinations = []

        for k in range(
            max(MIN_PHRASES_PER_EXAMPLE, 2),
            max_k + 1,
        ):
            all_combinations.extend(
                itertools.combinations(
                    range(n),
                    k,
                )
            )

        rng = random.Random(
            stable_seed(
                f"{RANDOM_SEED}:{condition}"
            )
        )

        rng.shuffle(all_combinations)

        for indices in all_combinations:
            if len(subsets) >= max_examples:
                break

            add_indices(indices)

    else:
        # Robust fallback for conditions with many symptom phrases.
        rng = random.Random(
            stable_seed(
                f"{RANDOM_SEED}:{condition}"
            )
        )

        attempts = 0
        max_attempts = max_examples * 100

        while (
            len(subsets) < max_examples
            and attempts < max_attempts
        ):
            attempts += 1

            k = rng.randint(
                max(MIN_PHRASES_PER_EXAMPLE, 2),
                max_k,
            )

            indices = tuple(
                sorted(
                    rng.sample(
                        range(n),
                        k,
                    )
                )
            )

            add_indices(indices)

    return subsets[:max_examples]


## 5. Generate candidate examples for all conditions

In [42]:
candidate_examples = []

per_condition_candidate_counts = {}

for record in conditions:
    condition = record["condition"]

    subsets = generate_phrase_subsets(
        condition,
        record["symptom_phrases"],
    )

    per_condition_candidate_counts[
        condition
    ] = len(subsets)

    for phrases in subsets:
        text_fields = build_example_text(
            phrases
        )

        candidate_examples.append(
            {
                "condition": condition,
                "symptom_phrases": phrases,
                "symptom_count": len(phrases),
                **text_fields,
            }
        )

print(
    "Candidate examples:",
    len(candidate_examples),
)

print(
    "Examples per condition - min:",
    min(per_condition_candidate_counts.values()),
)

print(
    "Examples per condition - max:",
    max(per_condition_candidate_counts.values()),
)

print(
    "Examples per condition - mean:",
    round(
        sum(per_condition_candidate_counts.values())
        / len(per_condition_candidate_counts),
        2,
    ),
)


Candidate examples: 8570
Examples per condition - min: 1
Examples per condition - max: 30
Examples per condition - mean: 22.79


## 6. Merge exact duplicate inputs into multi-label examples

A medical symptom input can legitimately match more than one condition.

If generated examples have the exact same normalized symptom input, we merge their labels:

```text
fatigue
→ ["Condition A", "Condition B", ...]
```

This avoids giving a classifier contradictory duplicate rows and keeps the dataset compatible with the project's **multi-label prediction** requirement.


In [43]:
grouped = {}

for example in candidate_examples:
    key = example[
        "symptoms_classical"
    ]

    if key not in grouped:
        grouped[key] = {
            "symptom_phrases": example[
                "symptom_phrases"
            ],
            "symptom_count": example[
                "symptom_count"
            ],
            "symptoms_canonical": example[
                "symptoms_canonical"
            ],
            "symptoms_classical": example[
                "symptoms_classical"
            ],
            "symptoms_tokens": example[
                "symptoms_tokens"
            ],
            "symptoms_bert": example[
                "symptoms_bert"
            ],
            "labels": set(),
        }

    grouped[key]["labels"].add(
        example["condition"]
    )


training_examples = []

for key in sorted(grouped):
    record = grouped[key]

    labels = sorted(
        record["labels"]
    )

    training_examples.append(
        {
            "example_id": (
                "ex_"
                + hashlib.sha256(
                    key.encode("utf-8")
                ).hexdigest()[:12]
            ),
            "symptom_phrases": record[
                "symptom_phrases"
            ],
            "symptom_count": record[
                "symptom_count"
            ],
            "symptoms_canonical": record[
                "symptoms_canonical"
            ],
            "symptoms_classical": record[
                "symptoms_classical"
            ],
            "symptoms_tokens": record[
                "symptoms_tokens"
            ],
            "symptoms_bert": record[
                "symptoms_bert"
            ],
            "labels": labels,
            "label_count": len(labels),
        }
    )


multi_label_examples = [
    r
    for r in training_examples
    if r["label_count"] > 1
]

print(
    "Unique training examples:",
    len(training_examples),
)

print(
    "Exact duplicate candidate rows merged:",
    len(candidate_examples)
    - len(training_examples),
)

print(
    "Multi-label examples:",
    len(multi_label_examples),
)


Unique training examples: 7849
Exact duplicate candidate rows merged: 721
Multi-label examples: 278


## 7. Validate label coverage before splitting

All model-ready conditions must still be represented somewhere in the generated examples.


In [44]:
expected_labels = set(
    condition_names
)

generated_labels = {
    label
    for example in training_examples
    for label in example["labels"]
}

missing_labels = sorted(
    expected_labels
    - generated_labels
)

unexpected_labels = sorted(
    generated_labels
    - expected_labels
)

print(
    "Expected labels:",
    len(expected_labels),
)

print(
    "Generated labels:",
    len(generated_labels),
)

print(
    "Missing labels:",
    len(missing_labels),
)

print(
    "Unexpected labels:",
    len(unexpected_labels),
)

assert not missing_labels, (
    "Some condition labels disappeared during example generation: "
    + ", ".join(missing_labels[:10])
)

assert not unexpected_labels

print("Label coverage passed ✅")


Expected labels: 376
Generated labels: 376
Missing labels: 0
Unexpected labels: 0
Label coverage passed ✅


## 8. Train / validation / test split

Target ratio:

```text
70% train
15% validation
15% test
```

### Leakage protection

We split **after exact duplicate inputs have been merged**, so the same normalized symptom text cannot appear in more than one split.

### Training-label protection

Every condition label is explicitly guaranteed to appear in the training set.

Validation/test coverage is reported rather than artificially duplicating examples across splits.


In [45]:
def deterministic_order(records, namespace):
    return sorted(
        records,
        key=lambda record: hashlib.sha256(
            (
                f"{RANDOM_SEED}:"
                f"{namespace}:"
                f"{record['example_id']}"
            ).encode("utf-8")
        ).hexdigest(),
    )


# Map each label to its examples.
label_to_examples = defaultdict(list)

for example in training_examples:
    for label in example["labels"]:
        label_to_examples[label].append(
            example
        )


# Step 1:
# Guarantee at least one training example for every label.
train_ids = set()

# Start with labels that have the fewest examples.
labels_by_rarity = sorted(
    expected_labels,
    key=lambda label: (
        len(label_to_examples[label]),
        label,
    ),
)

for label in labels_by_rarity:
    if any(
        example["example_id"] in train_ids
        for example in label_to_examples[label]
    ):
        continue

    options = deterministic_order(
        label_to_examples[label],
        f"train-coverage:{label}",
    )

    # Prefer an example with more symptom phrases.
    # It is generally more informative than a one-symptom example.
    options = sorted(
        options,
        key=lambda r: (
            -r["symptom_count"],
            r["example_id"],
        ),
    )

    train_ids.add(
        options[0]["example_id"]
    )


# Step 2:
# Fill the remaining global split targets deterministically.
total = len(training_examples)

target_train = round(
    total * TRAIN_RATIO
)

target_validation = round(
    total * VALIDATION_RATIO
)

target_test = (
    total
    - target_train
    - target_validation
)

remaining = [
    record
    for record in training_examples
    if record["example_id"]
    not in train_ids
]

remaining = deterministic_order(
    remaining,
    "global-split",
)

needed_train = max(
    0,
    target_train - len(train_ids),
)

additional_train = remaining[
    :needed_train
]

remaining = remaining[
    needed_train:
]

additional_train_ids = {
    record["example_id"]
    for record in additional_train
}

train_ids.update(
    additional_train_ids
)

validation_ids = {
    record["example_id"]
    for record in remaining[
        :target_validation
    ]
}

test_ids = {
    record["example_id"]
    for record in remaining[
        target_validation:
    ]
}


train_examples = [
    record
    for record in training_examples
    if record["example_id"] in train_ids
]

validation_examples = [
    record
    for record in training_examples
    if record["example_id"] in validation_ids
]

test_examples = [
    record
    for record in training_examples
    if record["example_id"] in test_ids
]


print("Total:", total)
print(
    "Train:",
    len(train_examples),
    f"({len(train_examples) / total:.1%})",
)
print(
    "Validation:",
    len(validation_examples),
    f"({len(validation_examples) / total:.1%})",
)
print(
    "Test:",
    len(test_examples),
    f"({len(test_examples) / total:.1%})",
)


Total: 7849
Train: 5494 (70.0%)
Validation: 1177 (15.0%)
Test: 1178 (15.0%)


## 9. Split validation and leakage checks

In [46]:
def labels_in(records):
    return {
        label
        for record in records
        for label in record["labels"]
    }


train_labels = labels_in(
    train_examples
)

validation_labels = labels_in(
    validation_examples
)

test_labels = labels_in(
    test_examples
)


missing_train_labels = sorted(
    expected_labels
    - train_labels
)

missing_validation_labels = sorted(
    expected_labels
    - validation_labels
)

missing_test_labels = sorted(
    expected_labels
    - test_labels
)


train_text = {
    record["symptoms_classical"]
    for record in train_examples
}

validation_text = {
    record["symptoms_classical"]
    for record in validation_examples
}

test_text = {
    record["symptoms_classical"]
    for record in test_examples
}


train_validation_overlap = (
    train_text
    & validation_text
)

train_test_overlap = (
    train_text
    & test_text
)

validation_test_overlap = (
    validation_text
    & test_text
)


all_split_ids = (
    {
        r["example_id"]
        for r in train_examples
    }
    | {
        r["example_id"]
        for r in validation_examples
    }
    | {
        r["example_id"]
        for r in test_examples
    }
)


print("SPLIT VALIDATION")
print("----------------")
print(
    "Training labels:",
    len(train_labels),
    "/",
    len(expected_labels),
)
print(
    "Labels missing from training:",
    len(missing_train_labels),
)
print(
    "Labels missing from validation:",
    len(missing_validation_labels),
)
print(
    "Labels missing from test:",
    len(missing_test_labels),
)
print()
print(
    "Train/validation exact-text overlap:",
    len(train_validation_overlap),
)
print(
    "Train/test exact-text overlap:",
    len(train_test_overlap),
)
print(
    "Validation/test exact-text overlap:",
    len(validation_test_overlap),
)


assert not missing_train_labels, (
    "Every condition label must appear in training."
)

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

assert len(all_split_ids) == len(
    training_examples
)

assert (
    len(train_examples)
    + len(validation_examples)
    + len(test_examples)
    == len(training_examples)
)

print("\nSplit validation passed ✅")


SPLIT VALIDATION
----------------
Training labels: 376 / 376
Labels missing from training: 0
Labels missing from validation: 52
Labels missing from test: 52

Train/validation exact-text overlap: 0
Train/test exact-text overlap: 0
Validation/test exact-text overlap: 0

Split validation passed ✅


## 10. Preview generated examples

Notice that `labels` is always a list, even when an example currently maps to only one condition. This keeps the dataset ready for multi-label models later.


In [47]:
PREVIEW_COUNT = 10

for example in training_examples[
    :PREVIEW_COUNT
]:
    print("=" * 90)
    print(
        "Example:",
        example["example_id"],
    )
    print(
        "Symptoms:",
        example["symptoms_canonical"],
    )
    print(
        "Labels:",
        example["labels"],
    )
    print(
        "Symptom count:",
        example["symptom_count"],
    )
    print()


Example: ex_a8a1db3209d6
Symptoms: a barking cough that sounds like a seal
Labels: ['Croup']
Symptom count: 1

Example: ex_1e3f0d747114
Symptoms: a barking cough that sounds like a seal ; a harsh sound (stridor) when breathing in
Labels: ['Croup']
Symptom count: 2

Example: ex_375359ca2ae5
Symptoms: a barking cough that sounds like a seal ; a harsh sound (stridor) when breathing in ; a hoarse voice
Labels: ['Croup']
Symptom count: 3

Example: ex_016f78844d29
Symptoms: a barking cough that sounds like a seal ; a harsh sound (stridor) when breathing in ; a hoarse voice ; difficulty breathing
Labels: ['Croup']
Symptom count: 4

Example: ex_e5abc1fa246a
Symptoms: a barking cough that sounds like a seal ; a harsh sound (stridor) when breathing in ; difficulty breathing
Labels: ['Croup']
Symptom count: 3

Example: ex_0de69657033f
Symptoms: a barking cough that sounds like a seal ; a hoarse voice
Labels: ['Croup']
Symptom count: 2

Example: ex_2c1b5de33a9f
Symptoms: a barking cough that sound

## 11. Build generation and split report

In [48]:
examples_per_label = Counter()

for example in training_examples:
    for label in example["labels"]:
        examples_per_label[label] += 1


report = {
    "random_seed": RANDOM_SEED,
    "source_condition_count": len(conditions),
    "candidate_example_count": len(candidate_examples),
    "unique_example_count": len(training_examples),
    "merged_duplicate_candidate_count": (
        len(candidate_examples)
        - len(training_examples)
    ),
    "multi_label_example_count": len(
        multi_label_examples
    ),
    "max_examples_per_condition": (
        MAX_EXAMPLES_PER_CONDITION
    ),
    "min_phrases_per_example": (
        MIN_PHRASES_PER_EXAMPLE
    ),
    "max_phrases_per_example": (
        MAX_PHRASES_PER_EXAMPLE
    ),
    "split_ratios": {
        "train": TRAIN_RATIO,
        "validation": VALIDATION_RATIO,
        "test": TEST_RATIO,
    },
    "split_counts": {
        "train": len(train_examples),
        "validation": len(
            validation_examples
        ),
        "test": len(test_examples),
    },
    "labels": {
        "total": len(expected_labels),
        "train": len(train_labels),
        "validation": len(
            validation_labels
        ),
        "test": len(test_labels),
        "missing_from_train": (
            missing_train_labels
        ),
        "missing_from_validation": (
            missing_validation_labels
        ),
        "missing_from_test": (
            missing_test_labels
        ),
    },
    "examples_per_label": dict(
        sorted(
            examples_per_label.items()
        )
    ),
    "leakage_checks": {
        "train_validation_exact_text_overlap": (
            len(train_validation_overlap)
        ),
        "train_test_exact_text_overlap": (
            len(train_test_overlap)
        ),
        "validation_test_exact_text_overlap": (
            len(validation_test_overlap)
        ),
    },
    "evaluation_note": (
        "Validation and test examples are source-derived "
        "symptom subsets from the same reviewed NHS condition "
        "records. They are useful for project comparison but "
        "are not an independent clinical evaluation set."
    ),
}

print(
    json.dumps(
        {
            "source_condition_count": report[
                "source_condition_count"
            ],
            "candidate_example_count": report[
                "candidate_example_count"
            ],
            "unique_example_count": report[
                "unique_example_count"
            ],
            "multi_label_example_count": report[
                "multi_label_example_count"
            ],
            "split_counts": report[
                "split_counts"
            ],
            "labels": {
                "total": report[
                    "labels"
                ]["total"],
                "train": report[
                    "labels"
                ]["train"],
                "validation": report[
                    "labels"
                ]["validation"],
                "test": report[
                    "labels"
                ]["test"],
            },
        },
        indent=2,
    )
)


{
  "source_condition_count": 376,
  "candidate_example_count": 8570,
  "unique_example_count": 7849,
  "multi_label_example_count": 278,
  "split_counts": {
    "train": 5494,
    "validation": 1177,
    "test": 1178
  },
  "labels": {
    "total": 376,
    "train": 376,
    "validation": 324,
    "test": 324
  }
}


## 12. Save training examples and splits

In [49]:
def save_json(path, data):
    with path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )


save_json(
    ALL_EXAMPLES_PATH,
    training_examples,
)

save_json(
    TRAIN_PATH,
    train_examples,
)

save_json(
    VALIDATION_PATH,
    validation_examples,
)

save_json(
    TEST_PATH,
    test_examples,
)

save_json(
    REPORT_PATH,
    report,
)


print("Saved:")
print("-", ALL_EXAMPLES_PATH)
print("-", TRAIN_PATH)
print("-", VALIDATION_PATH)
print("-", TEST_PATH)
print("-", REPORT_PATH)


Saved:
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\training\training_examples.json
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\training\train.json
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\training\validation.json
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\training\test.json
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\review\training_generation_report.json


## Training dataset ready ✅

The project now has a fixed dataset split that should be reused by all models:

```text
conditions_model_ready.json
        ↓
source-grounded example generation
        ↓
exact duplicate input merge
        ↓
multi-label examples
        ↓
70 / 15 / 15 split
        ↓
train.json
validation.json
test.json
```

### Next notebook

```text
5_tfidf_logistic_regression.ipynb
```

The TF-IDF baseline should use the **same saved split** as the later RNN, LSTM, and BERT/BioBERT models so their results are comparable.
